# `protein_based_graph_carey` — keep-fraction × criterion sweep (display only)

The Carey twin of `protein_based_graph.ipynb`: same approved pipeline GNN, same
display code, a **continuous** target. **This notebook only displays results.**
All numbers come from

    scripts/modeling/train/run_quantile_criteria_sweep.py --dataset cc

Three things differ from the M2OR notebook, all forced by the dataset:

1. **The x-axis is a keep FRACTION, not a coverage quantile.** Carey is a
   complete 50×110 receptor×odorant matrix, so every training molecule has
   identical coverage and `cov >= quantile(cov, q)` keeps all of them — on
   `our_inductive` both q=0.99 and q=0 give K=70 of 70. The sweep therefore runs
   `k_mode='fraction'`: keep the top (1−q) share outright. Same intent, tie-free
   reading. K is annotated on the plots because it, not q, is the real knob.
2. **Only `coverage` and `greedy_pair_cover`.** The other five criteria score
   molecules from positive/negative counts and are undefined on a z-score.
3. **Regression metrics.** R² is against the **test** mean; the naive predictor
   (train mean) sits at the dashed zero line, and on `our_inductive` its true
   value is −0.000 on every fold by construction — that is the whole point of
   that split family (upstream's `scaf` has a fold at −4.92).

Regimes here are split families: `inductive` → `our_inductive` (ours, stratified
cold molecule), `transductive` → `rand` (upstream i.i.d.).

In [ ]:
import sys, pathlib
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from scipy.stats import t as _t

ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / 'pyproject.toml').exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# --- what run_quantile_criteria_sweep.py --dataset cc produced ---
DATASET   = 'cc'   # 'hc' works too once its sweep has been run
QUANTILES = [0, 20, 40, 50, 60, 70, 80, 90]
N_MODELS  = 1      # bagging flag: 1 = the current pipeline default (no bagging), 5 = legacy bag
METR      = ['R2', 'RMSE', 'MAE', 'Pearson', 'Spearman']
LOWER_BETTER = {'RMSE', 'MAE'}
CACHE     = ROOT / 'results/graph/full_full/v7/protein_based_graph'

# regime -> (split family used in the filename, folds the sweep ran)
REGIMES = {
    'inductive':    ('our_inductive', [1, 2, 3, 4, 5]),
    'transductive': ('rand',          [1, 2, 3, 4, 5]),
}
MOL_SOURCES = {
    'ChemBERTa': f'chemberta_77m_{DATASET}',
    'GIN':       f'gin_supervised_contextpred_{DATASET}',
}

def csv_for(regime, source):
    key = f'{DATASET}_{REGIMES[regime][0]}'
    nm = '' if N_MODELS == 5 else f'__nm{N_MODELS}'   # 5-model bag = legacy name
    return CACHE / f'metrics_{key}__{MOL_SOURCES[source]}{nm}.csv'

# --- styling: criteria + boost reference line ---
# On a COMPLETE matrix `coverage`, `greedy_pair_cover`, `idf_coverage` and
# `composite` give every molecule the same score (see the header), so they act as
# tie-break controls rather than rankings -- keep one. `balance_bits`,
# `entropy_bits` and `disc_pairs` do rank, and rank IDENTICALLY here (Spearman
# 1.0000 / 0.9999 on cc/our_inductive), so one stands in for all three.
COL = {'coverage': '#2a78d6', 'greedy_pair_cover': '#4a3aa7',
       'balance_bits': '#eb6834', 'entropy_bits': '#1baf7a', 'disc_pairs': '#eda100',
       'idf_coverage': '#e87ba4', 'composite': '#008300', 'boost_full': '#0b0b0b'}
LABEL = {'boost_full': 'boost (full)'}
ORDER = ['coverage', 'greedy_pair_cover', 'balance_bits', 'boost_full']

def sty(c):
    if c == 'boost_full':
        return dict(ls='-', lw=2.6, marker='s', zorder=6)
    return dict(ls='-', lw=2.0, marker='o', zorder=3)

def mean_ci(vals):
    v = np.asarray(vals, float); v = v[np.isfinite(v)]; n = len(v)
    if n < 2:
        return (v.mean() if n else np.nan), 0.0
    return v.mean(), _t.ppf(0.975, n - 1) * v.std(ddof=1) / np.sqrt(n)

def _load(regime, source):
    csv = csv_for(regime, source)
    if not csv.exists():
        print(f'missing: {csv.name}  -- run the sweep for ({regime}, {source}) first')
        return None
    df = pd.read_csv(csv)
    if 'n_models' not in df.columns:
        df['n_models'] = 5              # legacy files predate the n_models column
    df = df[df['n_models'] == N_MODELS]
    if df.empty:
        print(f'{csv.name}: no rows at n_models={N_MODELS} (set N_MODELS to 1 or 5)')
        return None
    missing = [m for m in METR if m not in df.columns]
    if missing:
        print(f'{csv.name}: columns {missing} absent -- is this a classification sweep?')
        return None
    # Cells the sweep could not compute are stored as NaN with a reason. The one
    # that occurs here: at K=1 the single kept molecule can sit entirely on one
    # side of edge_threshold, and signed message passing needs both signs. Those
    # points are DROPPED, not zero-filled -- a missing model is not a bad model.
    bad = df[df[METR[0]].isna()]
    if len(bad):
        cols = ['criterion', 'quantile', 'K', 'seed'] + (['status'] if 'status' in bad.columns else [])
        print(f'{csv.name}: {len(bad)} degenerate cell(s) dropped')
        display(bad[cols])
        df = df[df[METR[0]].notna()]
    return df

def k_axis(df):
    '''q -> K, as the sweep resolved it. K is what actually changes the model.'''
    g = df[df.criterion != 'boost_full'].groupby('quantile')['K']
    return {float(q): int(v) for q, v in g.median().items()}

def value_fig(regime, source):
    '''metric value vs keep fraction, mean over folds with 95% CI.'''
    df = _load(regime, source)
    if df is None:
        return
    folds = REGIMES[regime][1]
    Kq = k_axis(df)
    _dodge = {c: (i - (len(ORDER) - 1) / 2) * 0.55 for i, c in enumerate(ORDER)}
    fig, axes = plt.subplots(2, 3, figsize=(16, 9))
    for ax, metric in zip(axes.ravel(), METR):
        for c in ORDER:
            sub = df[df.criterion == c]
            if sub.empty:
                continue
            qs = sorted(sub['quantile'].unique())
            means, errs = zip(*[mean_ci(sub[sub['quantile'] == q][metric].values) for q in qs])
            xd = [q + _dodge[c] for q in qs]; st = sty(c)
            ax.plot(xd, means, ms=5, color=COL[c], label=LABEL.get(c, c), **st)
            ax.errorbar(xd, means, yerr=errs, fmt='none', ecolor=COL[c],
                        elinewidth=1.1, capsize=2, alpha=0.3, zorder=st['zorder'] - 1)
        if metric == 'R2':
            # the naive predictor (train mean); on our_inductive it is -0.000 per fold
            ax.axhline(0.0, color='#999', ls='--', lw=1.2, zorder=1)
        ax.set_title(metric + (' (lower is better)' if metric in LOWER_BETTER else ''))
        ax.set_xlabel('keep fraction q  (K molecules kept)')
        ax.set_ylabel(metric); ax.set_xticks(QUANTILES)
        ax.set_xticklabels([f'{int(q)}\nK={Kq.get(float(q), "?")}' for q in QUANTILES], fontsize=8)
    for ax in axes.ravel()[len(METR):]:
        ax.axis('off')
    handles, labels = axes.ravel()[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc='upper center', ncol=len(ORDER),
               frameon=False, fontsize=8.5, bbox_to_anchor=(0.5, 1.03))
    fam = REGIMES[regime][0]
    fig.suptitle(f'{DATASET.upper()} GNN {source}-nodes (n_models={N_MODELS}) - {fam} - '
                 f'mean of folds {folds} (95% CI) - value vs keep fraction', y=1.06)
    fig.tight_layout(); plt.show()

def value_table(regime, source, q=90):
    '''Compact companion to value_fig: mean +- 95% CI per criterion at one q
    (rows = criteria + boost, cols = metrics), same shape as the ensembler table.'''
    df = _load(regime, source)
    if df is None:
        return
    sub = df[df['quantile'] == q]
    if sub.empty:
        print(f'{regime}/{source}: no rows at q {q}')
        return
    K = k_axis(df).get(float(q), '?')
    tbl = {}
    for c in ORDER:
        cc = sub[sub.criterion == c]
        if cc.empty:
            continue
        tbl[LABEL.get(c, c)] = {m: '%.3f +- %.3f (n=%d)' % (*mean_ci(cc[m].values),
                                                            int(cc[m].notna().sum()))
                                for m in METR}
    out = pd.DataFrame(tbl).T[METR]
    out.index.name = (f'{DATASET}/{REGIMES[regime][0]} / {source}  '
                      f'(q{q} -> K={K}, mean +- 95% CI, n_models={N_MODELS})')
    display(out)
    return out

def fold_table(regime, source, criterion='balance_bits', metric='R2'):
    '''Per-FOLD values. On these datasets a fold is a different set of molecules,
    not a reseed, so the spread is structural -- worth seeing, not just its CI.'''
    df = _load(regime, source)
    if df is None:
        return
    sub = df[df.criterion == criterion]
    if sub.empty:
        print(f'no rows for criterion {criterion!r}')
        return
    out = sub.pivot_table(index='quantile', columns='seed', values=metric)
    out.columns = [f'fold {c}' for c in out.columns]
    out['mean'] = out.mean(axis=1)
    out.index.name = f'{regime} / {source} / {criterion} - {metric} per fold'
    display(out.round(3))
    return out

plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': .25,
                     'axes.spines.top': False, 'axes.spines.right': False})
print('display setup ready; CACHE =', CACHE)

## Inductive (`our_inductive`) — the pipeline `cls+mol` feature

Cold molecule, stratified by response dynamic range so every fold is scorable.
K runs 35 → 1 of the 70 training molecules across q = 50 → 99.

### ChemBERTa nodes

In [ ]:
value_fig('inductive', 'ChemBERTa')
value_table('inductive', 'ChemBERTa')

In [ ]:
fold_table('inductive', 'ChemBERTa')

## Transductive (`rand`)

All 110 molecules are in train, so K runs 55 → 1. Upstream's i.i.d. family:
train and test means agree to three decimals, naive R² = −0.001.

### ChemBERTa nodes

In [ ]:
value_fig('transductive', 'ChemBERTa')
value_table('transductive', 'ChemBERTa')

In [ ]:
fold_table('transductive', 'ChemBERTa')